# Iteración QR para obtener la forma de Schur real.

In [1]:
using LinearAlgebra

La función **Housev** será necesaria para la descomposición de Hessenberg, mientras que la función **Givens** es útil para el paso QR.

In [2]:
function Housev(x)
    n = length(x)
    v = ones(size(x))
    v[2:n] = x[2:n]
    σ = norm(x[2:n])^2
    if σ == 0
        β = 0
    else 
        μ = √(x[1]^2+σ)
        if x[1] ≤ 0
            v[1] = x[1] - μ
        else
            v[1] = -σ/(x[1]+μ)
        end
        β = 2*v[1]^2/(σ+v[1]^2)
        v = v/(v[1])
    end
    return v, β
end


function Givens(a,b)
    if b==0
        c = 1
        s = 0
    else
        if abs(b)>abs(a)
            τ=-a/b
            s=-1/sqrt(1+τ^2)
            c=s*τ
        else
            τ=-b/a
            c=1/sqrt(1+τ^2)
            s=c*τ
        end
    end
    return c,s
end

Givens (generic function with 1 method)

Dada una matriz real $A$, la función **HessenbergForm** devuelve la descomposición de Hessenberg de $A$, es decir, matrices $H$ y $Q$ del mismo tamaño de $A$ tales que
$$
Q^TAQ = H
$$
donde $Q$ es ortogonal y $H$ es Hessenberg superior.

In [3]:
function HessenbergForm(A)
    n = size(A)[1]
    H = copy(A)
    Q = Matrix(1.0*I, n, n)
    for k = 1:n-2
        v, β = Housev(H[k+1:n, k])
        H[k+1:n, k:n] = (I - β*v*v')*H[k+1:n, k:n]
        H[1:n, k+1:n] = H[1:n, k+1:n]*(I - β*v*v')
        
        #Q es necesaria para la verificar que la función devuelve los resultados correctos
        Q[1:n, k+1:n] = Q[1:n, k+1:n]*(I - β*v*v') 
    end
    return H, Q
end

HessenbergForm (generic function with 1 method)

Verificamos la función con una matriz aleatoria cualquiera:

In [4]:
B = floor.(10*rand(5,5)-10*I)

H, Q = HessenbergForm(B)
println("|Q'Q-I|=",opnorm(Q'Q-I))
println("|Q'BQ-H|=",opnorm(Q'*B*Q-H))
println("\n H es")
display(H)

|Q'Q-I|=5.481015151666629e-16
|Q'BQ-H|=8.135194721614724e-15

 H es


5×5 Matrix{Float64}:
 -7.0           3.08062       5.61942       2.57122    -2.7057
  7.14143       2.7451        7.64191      -0.576712   -5.87515
 -3.33067e-16   4.87427      -3.23186      -3.41737    -1.05279
 -1.11022e-16  -1.85856e-15   2.32351      -9.68033    -4.73666
 -2.22045e-16   1.21173e-17   3.03679e-17  -2.8862    -10.8329

La función **HessenbergQR** efectúa un paso de la iteración $QR$. Toma $H$ Hessenber superior, efectua la factorización $QR$ de $H$, de manera que 
$$H=QR$$
y devuelve la matriz $RQ$, la cual resulta Hessenberg superior y la cual se utiliza en la siguiente iteración.

In [5]:
function HessenbergQR(H)
    n = size(H)[1]
    H2 = copy(H)
    C, S = zeros(n-1), zeros(n-1)
    
    #Factorización QR de H
    for k = 1:n-1
        C[k], S[k] = Givens(H2[k,k], H2[k+1, k])
        H2[k:k+1,k:n] = [C[k] -S[k]; S[k] C[k]]*H2[k:k+1,k:n]
        H2[k+1,k] = 0
        #display([C[k] S[k]])
    end
    
    #Matriz RQ
    for k = 1:n-1
        H2[1:k+1, k:k+1] = H2[1:k+1, k:k+1]*[C[k] S[k]; -S[k] C[k]]
    end
    
    return H2 #RQ Hessenberg superior
end


HessenbergQR (generic function with 1 method)

In [49]:
min([1,2,3]...)

1

**RealSchur** toma la matriz $A$, calcula su forma de Hessenberg y luego aplica varias iteraciones $QR$. La función se detiene cuando se alcanza cierta precisión o cuando se alcanza el máximo de iteraciones.

In [137]:
function RealSchur(A, iteraciones = 10000)
    H0 = A
    H1 = HessenbergForm(A)[1]
    δ = 10
    for k = 1:iteraciones
        H0 = H1
        H1 = HessenbergQR(H1)
    end
    return H1
    
end

RealSchur (generic function with 3 methods)

Las matrices de rango 1 tienen dos autovalores reales.

In [138]:
x = rand(5)
B = x*x'
println("Los autovalores de B son:")
display(eigvals(B))
println("\n El resultado de la iteración es")
display(RealSchur(B))

Los autovalores de B son:


5-element Vector{Float64}:
 -3.8474835855695245e-17
  3.1005778097314713e-19
  5.106657403158995e-18
  1.8595538421316525e-16
  1.5475009653543446


 El resultado de la iteración es


5×5 Matrix{Float64}:
 1.5475        1.66667e-16   8.54369e-17  -7.96292e-17  -1.56047e-17
 0.0          -6.9679e-17    1.98863e-16  -9.82421e-18  -1.96079e-18
 1.00106e-16   0.0           4.62353e-17   4.48404e-17   1.4416e-18
 2.6968e-16   -1.28812e-31  -1.21298e-17   2.68944e-17  -3.60569e-18
 1.59087e-17  -2.60475e-33   2.71219e-34   0.0           9.39843e-19

Otro ejemplo con una matriz con autovalores reales:

In [139]:
B = [-6.0   91.0   77.0   23.0    5.0
     30.0  -79.0   82.0   93.0   62.0
     79.0    4.0  -41.0   95.0   40.0
      3.0   13.0   63.0  -29.0   57.0
     84.0   53.0   83.0   73.0  -97.0]
println("Los autovalores son:")
display(eigvals(B))
println("El resultado de la iteración QR es")
display(RealSchur(B))

Los autovalores son:


5-element Vector{Float64}:
 -127.49299726129188
 -123.74122853300453
 -102.7677706611078
  -63.14852144578727
  165.1505179011917

El resultado de la iteración QR es


5×5 Matrix{Float64}:
 165.151          54.8879         47.6412        -60.1161    -51.4828
  -1.26e-321    -127.493         -12.5276        -22.6494    -52.9317
  -1.68754e-14     2.5481e-130  -123.741          49.0362    -66.7352
  -2.77556e-17     5.94514e-15    -1.02e-321    -102.768     -64.6791
  -6.66134e-15    -5.94401e-15     2.93248e-17    -3.1e-322  -63.1485

Para los ejemplos con matrices complejas implementamos una función que permita ver los autovalores de los bloques $2\times2$ 

In [140]:
function autovComplejos(H)
    D = diag(H,-1)
    for k = 1:size(D)[1]
        if abs(D[k])>10^-12
            display(eigvals(H[k:k+1,k:k+1]))
        end
    end
end

autovComplejos (generic function with 1 method)

In [143]:
#Ejemplo con matrices aleatorias
E = floor.(100*rand(5,5)-I*100)
display(E)
print("Los autovalores complejos son: \n")
display(eigvals(E))

print("La forma de Schur real es: \n")
reSchurE = RealSchur(E)
display(reSchurE)

print("Los autovalores de los bloques 2x2 son: \n")
autovComplejos(reSchurE)

5×5 Matrix{Float64}:
 -54.0   97.0   56.0   44.0   84.0
  87.0  -17.0   27.0   49.0   27.0
  96.0   22.0  -36.0    6.0   86.0
  59.0   43.0   15.0  -71.0   86.0
  90.0   54.0   24.0   89.0  -26.0

Los autovalores complejos son: 


5-element Vector{ComplexF64}:
  -158.1018038541052 + 0.0im
  -136.7755306107281 + 0.0im
 -51.384715440903335 - 13.956608593603189im
 -51.384715440903335 + 13.956608593603189im
  193.64676534664005 + 0.0im

La forma de Schur real es: 


5×5 Matrix{Float64}:
 193.647         -19.4            41.607       -26.2865   -13.5555
   1.56e-321    -158.102          -6.52125      -2.62194    7.30765
  -3.77476e-15    -2.026e-321   -136.776       -30.3755   -43.5178
   8.99281e-15    -7.17075e-16     0.0         -51.0677    14.9879
  -2.22045e-15     4.14906e-18     1.8606e-15  -13.003    -51.7018

Los autovalores de los bloques 2x2 son: 


2-element Vector{ComplexF64}:
 -51.384715440902724 - 13.956608593603095im
 -51.384715440902724 + 13.956608593603095im